# 01 · Dataset audit
> **Research prototype.** Predicts *dataset-defined* dementia categories from preprocessed JPG brain MRI slices. Not a clinically validated diagnostic system.

Steps 1–11 of the execution order: locate/download the dataset, inspect the structure, discover the classes from the
folders (nothing is hard-coded), verify every image, compute MD5/SHA256, exact and perceptual duplicates, original /
augmented filename relationships and patient / study / examination identifiers. **No training happens here.**

In [ ]:
# Bootstrap: locate the project code (local checkout, /kaggle/working, or a Kaggle dataset upload)
import os, sys, glob, shutil, json
from pathlib import Path

CANDIDATES = [Path.cwd().parent, Path.cwd(), Path("/kaggle/working/dementia_mri_project")]
CANDIDATES += [Path(p).parent for p in glob.glob("/kaggle/input/*/dementia_mri_project/src")
               + glob.glob("/kaggle/input/*/*/dementia_mri_project/src") + glob.glob("/kaggle/input/*/src")]
CODE = next((c for c in CANDIDATES if (c / "src" / "run_pipeline.py").exists()), None)
if CODE is None:
    raise FileNotFoundError("Project code not found. On Kaggle: upload dementia_mri_project.zip as a Dataset and add it "
                            "as an Input (Kaggle extracts it), or put the project folder next to this notebook.")
if str(CODE).startswith("/kaggle/input"):                       # /kaggle/input is read-only -> work in /kaggle/working
    dst = Path("/kaggle/working/dementia_mri_project")
    shutil.copytree(CODE / "src", dst / "src", dirs_exist_ok=True)
    shutil.copy(CODE / "predict.py", dst / "predict.py")
    if not (dst / "configs" / "config.json").exists():
        shutil.copytree(CODE / "configs", dst / "configs", dirs_exist_ok=True)
    for sub in ("data", "checkpoints", "results", "gradcam", "reports"):   # outputs of earlier notebooks, if attached
        if (CODE / sub).exists() and not (dst / sub).exists():
            shutil.copytree(CODE / sub, dst / sub)
    CODE = dst
sys.path.insert(0, str(CODE)); os.chdir(CODE)
print("Project code:", CODE)

from src.utils import load_config, get_paths, print_env, load_json, save_json
from src.nb_helpers import show, show_md, show_table, sample_grid

CONFIG_OVERRIDES = {}      # e.g. {"batch_size": 16, "num_workers": 4, "epochs_stage2": 30}
cfg = load_config(overrides=CONFIG_OVERRIDES)
paths = get_paths(cfg)
ENV = print_env()
print("Outputs ->", paths["root"])

In [ ]:
import pandas as pd
from src.run_pipeline import stage_audit
AUDIT, SUMMARY = stage_audit(cfg, paths)

## Class mapping (verified from the folders) and headline numbers

In [ ]:
print(json.dumps(load_json(paths["data"] / "class_mapping.json"), indent=1))
for k in ("total_files", "valid_images", "corrupted_images", "unlabelled_images", "final_usable_images_before_dedup",
          "class_distribution", "imbalance_ratio_max_min", "rgb_channels_nearly_identical_frac"):
    print(f"{k:<36} {SUMMARY[k]}")

## Class distribution and image statistics

In [ ]:
show(paths["results"] / "class_distribution.png")
show(paths["results"] / "image_statistics_by_class.png")
valid = AUDIT[AUDIT.valid.fillna(False).astype(bool)]
display(valid.groupby("class")[["width", "height", "brightness", "contrast", "file_size", "black_frac",
                                "max_channel_diff_p99", "jpeg_quality_est"]].describe().T.round(2))

## Example images per class

In [ ]:
classes = [c["name"] for c in load_json(paths["data"] / "class_mapping.json")["classes"]]
sample_grid(valid, classes, 6, title="Random valid images per class")

## Corrupted images, exact duplicates, perceptual duplicates

In [ ]:
show_table(paths["audit"] / "corrupted_images.csv")
show_table(paths["audit"] / "duplicate_groups.csv")
nd = pd.read_csv(paths["audit"] / "near_duplicate_report.csv")
print(nd.decision.value_counts())
display(nd.sort_values("hamming").head(15)[["image_a", "image_b", "hamming", "similarity_score", "transform", "pixel_corr", "decision", "reason"]])

## Original / augmented names and identifiers

In [ ]:
print(json.dumps(SUMMARY["identifiers"], indent=1, default=str))
display(valid[["relpath", "patient_id", "study_id", "exam_id", "is_augmented", "aug_evidence", "base_key"]].head(15))

## Audit report

In [ ]:
show_md(paths["reports"] / "dataset_audit.md")